# Basic 2 mT5-Base FOLIO Training On Kaggle T4 x2

Goal: fine-tune `google/mt5-base` on FOLIO for natural language -> FOL generation. Do not run the final symbolic pipeline here.

## A. Environment Check And Dependencies

In [ ]:
!pip install -q -U transformers datasets accelerate sentencepiece evaluate


In [ ]:
import json
import os
import shutil
import subprocess
import tarfile
from pathlib import Path

import torch

print('torch.cuda.is_available() =', torch.cuda.is_available())
print('torch.cuda.device_count() =', torch.cuda.device_count())
if torch.cuda.device_count() < 1:
    raise RuntimeError('Basic 2 real training requires a Kaggle GPU. Enable GPU accelerator.')
if torch.cuda.device_count() == 1:
    print('WARNING: only 1 GPU detected. T4 x2 workflow expects 2 GPUs.')
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
print('RANK =', os.environ.get('RANK'))
print('LOCAL_RANK =', os.environ.get('LOCAL_RANK'))
print('WORLD_SIZE =', os.environ.get('WORLD_SIZE'))


## B. Copy Repo/Code If Needed

In [ ]:
def find_repo_root():
    candidates = [
        Path('/kaggle/working/Neurosymbolic-Reasoning'),
        Path('/kaggle/working/neurosymbolic-reasoning'),
        Path.cwd(),
        Path('/kaggle/input/neurosymbolic-reasoning'),
    ]
    for p in candidates:
        if (p / 'scripts' / 'train_basic2_mt5_folio.py').exists():
            return p
    raise FileNotFoundError('Could not find repo root with scripts/train_basic2_mt5_folio.py')

repo_root = find_repo_root()
if str(repo_root).startswith('/kaggle/input'):
    target = Path('/kaggle/working/Neurosymbolic-Reasoning')
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(repo_root, target)
    repo_root = target

os.chdir(repo_root)
print('REPO_ROOT =', repo_root)
print('cwd =', Path.cwd())
print('train script exists =', Path('scripts/train_basic2_mt5_folio.py').exists())


## C. Dry-Run Dataset Check

In [ ]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError('None of these paths exist:\n' + '\n'.join(map(str, paths)))

TRAIN_FILE = first_existing([
    '/kaggle/input/neurosymbolic-reasoning/dataset/filtered/train/folio_train.json',
    'dataset/filtered/train/folio_train.json',
])
VALID_FILE = first_existing([
    '/kaggle/input/neurosymbolic-reasoning/dataset/filtered/valid/folio_valid.json',
    'dataset/filtered/valid/folio_valid.json',
])
TEST_FILE = first_existing([
    '/kaggle/input/neurosymbolic-reasoning/dataset/full/Folio/folio_test.json',
    'dataset/full/Folio/folio_test.json',
])

def audit_json(path):
    rows = json.loads(Path(path).read_text(encoding='utf-8'))
    print('\n', path)
    print('rows =', len(rows))
    print('first keys =', list(rows[0].keys()) if rows else [])
    return rows

train_rows = audit_json(TRAIN_FILE)
valid_rows = audit_json(VALID_FILE)
test_rows = audit_json(TEST_FILE)
print('Resolved TRAIN_FILE =', TRAIN_FILE)
print('Resolved VALID_FILE =', VALID_FILE)
print('Resolved TEST_FILE =', TEST_FILE)


## D. Launch Real Basic 2 Training With Accelerate

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/outputs/basic2_mt5_base_folio')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    'accelerate', 'launch', '--num_processes', '2', 'scripts/train_basic2_mt5_folio.py',
    '--profile', 'base',
    '--train-file', str(TRAIN_FILE),
    '--valid-file', str(VALID_FILE),
    '--output-dir', str(OUTPUT_DIR),
    '--model-name', 'google/mt5-base',
    '--max-source-length', '768',
    '--max-target-length', '768',
    '--num-train-epochs', '20',
    '--learning-rate', '5e-5',
    '--per-device-train-batch-size', '1',
    '--per-device-eval-batch-size', '1',
    '--gradient-accumulation-steps', '8',
    '--eval-steps', '25',
    '--save-steps', '25',
    '--logging-steps', '5',
    '--save-total-limit', '3',
    '--fp16',
    '--gradient-checkpointing',
    '--optim', 'adafactor',
    '--predict-with-generate',
    '--generation-max-length', '768',
    '--seed', '42',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## OOM Fallback Command Before mt5-small

In [ ]:
fallback_cmd = [
    'accelerate', 'launch', '--num_processes', '2', 'scripts/train_basic2_mt5_folio.py',
    '--profile', 'base', '--train-file', str(TRAIN_FILE), '--valid-file', str(VALID_FILE),
    '--output-dir', '/kaggle/working/outputs/basic2_mt5_base_folio_len512',
    '--model-name', 'google/mt5-base', '--max-source-length', '512', '--max-target-length', '512',
    '--num-train-epochs', '20', '--learning-rate', '5e-5',
    '--per-device-train-batch-size', '1', '--per-device-eval-batch-size', '1',
    '--gradient-accumulation-steps', '16', '--eval-steps', '25', '--save-steps', '25',
    '--logging-steps', '5', '--save-total-limit', '3', '--fp16', '--gradient-checkpointing',
    '--optim', 'adafactor', '--predict-with-generate', '--generation-max-length', '512', '--seed', '42'
]
print('If the 768-length base run OOMs, run this before switching to mt5-small:')
print(' '.join(fallback_cmd))


## E. Find Best Checkpoint

In [ ]:
def find_best_checkpoint(output_dir):
    output_dir = Path(output_dir)
    states = sorted(output_dir.glob('checkpoint-*/trainer_state.json'))
    for state_path in reversed(states):
        state = json.loads(state_path.read_text(encoding='utf-8'))
        best = state.get('best_model_checkpoint')
        if best and Path(best).exists():
            return Path(best)
    checkpoints = sorted(output_dir.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
    return checkpoints[-1] if checkpoints else output_dir

BEST_CHECKPOINT = find_best_checkpoint(OUTPUT_DIR)
FINAL_CHECKPOINT = OUTPUT_DIR
print('BEST_CHECKPOINT =', BEST_CHECKPOINT)
print('FINAL_CHECKPOINT =', FINAL_CHECKPOINT)


## F. Generate Sanity Examples

In [ ]:
sample_output = OUTPUT_DIR / 'sample_generations.json'
subprocess.run([
    'python', 'scripts/test_basic2_mt5_checkpoint.py',
    '--checkpoint', str(BEST_CHECKPOINT),
    '--valid-file', str(VALID_FILE),
    '--output', str(sample_output),
    '--num-examples', '5',
], check=True)
print('sample generations =', sample_output)


## G. Write Handoff README And Optional Tarball

In [ ]:
readme_path = OUTPUT_DIR / 'README_CHECKPOINT.md'
training_command = ' '.join(cmd)
resume_command = '''accelerate launch --num_processes 2 scripts/train_basic2_mt5_folio.py \\
  --profile base \\
  --train-file dataset/filtered/train/folio_train.json \\
  --valid-file dataset/filtered/valid/folio_valid.json \\
  --output-dir outputs/basic2_mt5_base_folio_continued \\
  --model-name <downloaded_checkpoint_or_google/mt5-base> \\
  --resume-from-checkpoint <downloaded_checkpoint_path> \\
  --max-source-length 768 \\
  --max-target-length 768 \\
  --num-train-epochs 30 \\
  --learning-rate 5e-5 \\
  --per-device-train-batch-size 1 \\
  --per-device-eval-batch-size 1 \\
  --gradient-accumulation-steps 8 \\
  --eval-steps 25 \\
  --save-steps 25 \\
  --logging-steps 5 \\
  --save-total-limit 3 \\
  --fp16 \\
  --gradient-checkpointing \\
  --optim adafactor \\
  --predict-with-generate \\
  --generation-max-length 768 \\
  --seed 42'''
readme_path.write_text(f'''# Basic 2 mT5-Base FOLIO Checkpoint

- model: google/mt5-base
- hardware: Kaggle T4 x2
- train file: {TRAIN_FILE}
- valid file: {VALID_FILE}
- epochs: 20
- per-device batch size: 1
- gradient accumulation: 8
- effective batch size: 16 across 2 GPUs
- learning rate: 5e-5
- best checkpoint path: {BEST_CHECKPOINT}
- final checkpoint path: {FINAL_CHECKPOINT}
- sample generation file: {sample_output}

## Training Command

```bash
{training_command}
```

## Resume Command For Cảnh

```bash
{resume_command}
```
''', encoding='utf-8')
print('README =', readme_path)

tar_path = Path('/kaggle/working/basic2_mt5_base_best_checkpoint.tar.gz')
try:
    with tarfile.open(tar_path, 'w:gz') as tar:
        tar.add(BEST_CHECKPOINT, arcname=BEST_CHECKPOINT.name)
        tar.add(readme_path, arcname='README_CHECKPOINT.md')
    print('tarball =', tar_path)
except Exception as exc:
    print('Could not create tarball; download output folder from Kaggle instead:', exc)
